# Darija Tutor - Full 3,000-Row Generation (v3)

Third attempt. Read this cell before running anything - **how** you launch
this notebook matters more than anything in it.

## Run this with "Save & Run All (Commit)", NOT interactively

Both previous runs died the same way, and it was never the generation code:

| Run | Launched | Died | Unattended for |
|---|---|---|---|
| v1 (first freeze) | 11:20:47 | ~12:49 | ~89 min |
| v4.1 (last night) | 03:20:29 | ~04:20:20 | **59 min 45 s** |

The v4.1 number is the giveaway: **59:45 is Kaggle's 60-minute interactive
idle timeout**. An interactive session with no browser activity gets the
"are you still there?" prompt and is then stopped - taking every process in
the container with it, including detached ones. No amount of
`start_new_session=True`, watchdogs, or socket timeouts can survive the
container being stopped out from under them.

**The fix is to not use an interactive session.** Use
`Save Version -> Save & Run All (Commit)`. A committed run executes
headless in batch, has no idle timeout at all (there is no browser to be
idle), gets the full 12-hour budget, and persists `/kaggle/working` as
version output. You close the tab and come back later - that is the
supported way to run a multi-hour job.

A JS keep-alive hack to fake browser activity is not a fix; batch mode is
the mechanism that actually exists for this.

## Consequence for this notebook's structure

In batch mode, when the last cell finishes, Kaggle tears the container
down. So the monitor cell in Section 5 **blocks until generation
completes** - it is what keeps the session alive. Do not remove it or
"optimise" it into a non-blocking check, or the container will be
destroyed seconds after the generators launch.

## Code fixes since v4.1

- **`arabic_script` "42%" was a reporting bug, not a data defect.** With
  `--resume`, `generated_count` was seeded from the banked rows but
  `arabic_script_count` started at 0, so the ratio was
  `new_arabic / (resumed + new)` = 167/400. Every row on disk from that run
  is Arabic script and passes the code-switching gate. Numerator is now
  seeded too, and the summary line reports resumed vs. new separately.
- **Cold-start timeout.** `chunk_timeout=60` was applied to the first read,
  but Ollama sends nothing while loading the 5.8GB model - so all four
  workers timed out at exactly 60s on the first wave of the v4.1 run.
  Time-to-first-token now has its own `first_chunk_timeout=300` budget,
  tightened to `chunk_timeout` once the first byte arrives.
- **`poll()` cannot be trusted** to detect a dead child (see Section 5).

## Environment gaps found by manually running v3 on Kaggle

A hand-run of the previous version needed live tweaks to get through setup
at all - not the batch/idle-timeout issue above, just Kaggle-image
realities that were missing from the notebook. All four are now baked in:

- **Dataset mount path varies.** This upload showed up at
  `/kaggle/input/datasets/<username>/<dataset-slug>/`, not the flat
  `/kaggle/input/<dataset-slug>/` assumed before. Section 1 now searches
  for the uploaded package under `/kaggle/input` instead of requiring a
  hardcoded guess.
- **`tests/` was never copied**, so the pre-flight's `pytest tests/` had
  nothing to run against. Added to the copy list.
- **Ollama's installer needs `zstd`** to unpack `ollama-linux-amd64.tar.zst`
  - not present on the base image. Installed before the installer runs.
- **A freshly-started `ollama serve` can fail to come up on the first
  try.** One GPU needed its model-storage directories created by hand and
  the server relaunched. Section 3 now pre-creates those directories and
  retries the launch automatically instead of requiring a human to notice
  and intervene - which defeats the entire point of running this
  unattended in batch mode.

**Settings:** Accelerator: GPU T4 x2 - Internet: On - Persistence:
Variables and Files.

**Before running:** upload a Kaggle Dataset containing `app/`, `data/`,
`raw/`, `tests/`, and the banked rows you want to resume from.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Locate the dataset, copy it in, seed banked rows

`/kaggle/input/...` is read-only; the pipeline writes into
`/kaggle/working/`. Kaggle's actual mount path for an uploaded dataset is
not fixed - a flat `/kaggle/input/<dataset-slug>/` and a nested
`/kaggle/input/datasets/<username>/<dataset-slug>/` have both been observed
on this project, and hardcoding either one breaks on the other. So this
searches for the uploaded package instead of assuming a path.

`BANKED` should point at the directory holding the `out_full_gpu0/` and
`out_full_gpu1/` shards you want `--resume` to count (the incident export
has 400 + 387 verified `socratic` rows, 100% Arabic script and gate-passing
- see `opus_kagglefreeze_fix.md`). Leave `BANKED_SUBDIR` empty to start
completely fresh instead.


In [ ]:
import os
from pathlib import Path

def discover_src(marker="app/services/generate_training_data.py"):
    """Find the uploaded dataset root under /kaggle/input by locating a
    known file inside it, instead of assuming a fixed mount path."""
    for root, dirs, files in os.walk("/kaggle/input"):
        candidate = Path(root) / marker
        if candidate.is_file():
            return Path(root)
    return None

SRC = discover_src()
if SRC is None:
    print("Could not auto-locate the dataset under /kaggle/input. Contents:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        print("  " * depth + root)
        for f in files[:5]:
            print("  " * (depth + 1) + f)
        if depth > 4:
            break
    raise SystemExit("Set SRC manually from the listing above, then re-run this cell.")
print("SRC =", SRC)

BANKED_SUBDIR = "incident_export"   # <-- set to "" to start socratic fresh
BANKED = SRC / BANKED_SUBDIR if BANKED_SUBDIR else None


In [ ]:
import shutil
from pathlib import Path

# tests/ must be copied too - the pre-flight cell below runs pytest against
# it, and a missing tests/ silently left it with nothing to check.
for d in ["app", "data", "raw", "tests"]:
    shutil.copytree(f"{SRC}/{d}", f"/kaggle/working/{d}", dirs_exist_ok=True)
os.chdir("/kaggle/working")

RAW_GLOB = "*_raw.jsonl"
for gpu in (0, 1):
    dst = Path(f"/kaggle/working/out_full_gpu{gpu}")
    dst.mkdir(parents=True, exist_ok=True)
    if not BANKED:
        print(f"GPU{gpu}: starting fresh (BANKED=None)")
        continue
    src_dir = Path(BANKED) / f"out_full_gpu{gpu}"
    if not src_dir.exists():
        print(f"GPU{gpu}: no banked shard at {src_dir} - starting fresh")
        continue
    for f in sorted(src_dir.glob(RAW_GLOB)):
        if f.stat().st_size == 0:
            print(f"GPU{gpu}: skipping empty {f.name}")
            continue
        shutil.copy(f, dst / f.name)
        n = sum(1 for _ in open(dst / f.name, encoding="utf-8"))
        print(f"GPU{gpu}: seeded {n:4d} rows from {f.name}")

!ls raw/shared/*/text | head -5


## 2. Pre-flight - fail fast, before spending GPU time

Verifies the fixed pipeline is what actually got uploaded. The
`first_chunk_timeout` check is specifically confirming the cold-start fix
is present, since its absence only shows up as four mysterious timeouts a
minute into the run.


In [ ]:
!python -m py_compile app/services/generate_training_data.py && echo "compiles OK"
import pathlib
if pathlib.Path("tests").is_dir():
    !python -m pytest tests/ -q
else:
    print("tests/ not present in this upload - skipping (add it to the Kaggle Dataset to enable)")

import inspect, sys
sys.path.insert(0, "/kaggle/working")
from app.services.generate_training_data import call_ollama, generate_component
sig = inspect.signature(call_ollama)
assert "first_chunk_timeout" in sig.parameters, "cold-start fix MISSING - re-upload app/"
src = inspect.getsource(generate_component)
assert "resume_rows or []" in src, "resume arabic_script fix MISSING - re-upload app/"
print("cold-start fix present :", sig.parameters["first_chunk_timeout"])
print("resume stats fix present: yes")


## 3. Ollama - install, launch one server per GPU, wait for readiness

`ollama serve` failed to come up on its first launch for one GPU during the
manual run this was built from - a fresh container's
`/root/.ollama/models/{manifests,blobs}` didn't exist yet, and the server
did not survive the resulting race. Two changes: those directories are
created up front so the race can't happen, and the launch is wrapped in a
bounded retry loop so a transient failure doesn't require a human to
notice a dead port and relaunch it by hand - the entire justification for
batch mode is that nobody is watching.


In [ ]:
# ollama's installer unpacks a .tar.zst archive; zstd is not on the base image.
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, os, time, urllib.request

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

MODEL = "hf.co/QuantFactory/Atlas-Chat-9B-GGUF:Q4_K_M"
PORTS = {0: 11434, 1: 11435}

base_env = dict(
    os.environ,
    OLLAMA_NUM_PARALLEL="4",
    OLLAMA_MAX_LOADED_MODELS="1",
    OLLAMA_KEEP_ALIVE="60m",
)

os.makedirs("/root/.ollama/models/manifests", exist_ok=True)
os.makedirs("/root/.ollama/models/blobs", exist_ok=True)

def wait_for_server(port, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=5)
            return True
        except Exception:
            time.sleep(3)
    return False

def launch_ollama(gpu, port, attempts=3):
    env = dict(base_env, CUDA_VISIBLE_DEVICES=str(gpu), OLLAMA_HOST=f"127.0.0.1:{port}")
    for attempt in range(1, attempts + 1):
        # Append, never truncate: a fresh "w" on every retry is what left
        # ollama_gpu0.log at 0 bytes after the last incident - the retry
        # wiped whatever the failed first attempt had written.
        log = open(f"/kaggle/working/ollama_gpu{gpu}.log", "a")
        proc = subprocess.Popen(
            ["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        if wait_for_server(port):
            print(f"GPU{gpu} (port {port}): up on attempt {attempt}, pid {proc.pid}")
            return proc
        print(f"GPU{gpu} (port {port}): attempt {attempt}/{attempts} did not "
              f"come up within 60s, retrying...")
        if proc.poll() is None:
            proc.terminate()
        time.sleep(5)
    raise RuntimeError(f"ollama on GPU{gpu} never became ready after {attempts} attempts")

ollama_procs = {gpu: launch_ollama(gpu, port) for gpu, port in PORTS.items()}


In [ ]:
# Pull once per server, then confirm the model is actually visible on both -
# belt-and-suspenders check added after a run where this was assumed rather
# than verified. Both share ~/.ollama, so the second pull is a cache hit.
for gpu, port in PORTS.items():
    env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}")
    subprocess.run(["ollama", "pull", MODEL], env=env, check=True)

for gpu, port in PORTS.items():
    tags = urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=10).read().decode()
    assert MODEL in tags, f"GPU{gpu}: {MODEL} not visible on port {port} after pull"
    print(f"GPU{gpu} (port {port}): model confirmed present")


## 4. Launch generation on both GPUs

`--resume` counts the seeded rows toward each component's target and seeds
the dedup set from them, so nothing already banked is regenerated or
duplicated.


In [ ]:
def launch_generator(gpu):
    cmd = [
        "python", "-u", "-m", "app.services.generate_training_data",
        "--target-rows", "1500",
        "--concurrency", "4",
        "--model", MODEL,
        "--ollama-url", f"http://127.0.0.1:{PORTS[gpu]}",
        "--script-policy", "allow",
        "--log-level", "INFO",
        "--output-dir", f"/kaggle/working/out_full_gpu{gpu}",
        "--resume",
    ]
    log_fp = open(f"/kaggle/working/gen_full_gpu{gpu}.log", "a")
    return subprocess.Popen(
        cmd, stdout=log_fp, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True, cwd="/kaggle/working",
    )

gen_procs = {gpu: launch_generator(gpu) for gpu in PORTS}
print("generation launched:", {g: p.pid for g, p in gen_procs.items()})


## 5. Blocking monitor - THIS CELL KEEPS THE BATCH SESSION ALIVE

Do not make this non-blocking. In a committed run, Kaggle destroys the
container when the notebook's cells finish; if this returns immediately the
generators are killed seconds after launch.

**Why it does not use `p.poll()`:** `poll()` reports a *phantom clean exit*.
CPython's `Popen._try_wait` catches `ChildProcessError` (ECHILD - waitpid on
a PID that is not your child) and sets the status to `0`. So after a session
restart, or any time the child has been reparented, `poll()` returns `0` and
the run looks like it "completed successfully". That is exactly what
happened after the v4.1 run: both processes reported exit code 0 while
`code_switching_raw.jsonl` was 0 bytes and `socratic` was 15 rows short.
`os.kill(pid, 0)` asks the kernel whether the PID exists instead of asking
about parentage, and the `/proc` cmdline check guards against PID reuse.

### How to actually check in on a committed run

You cannot run ad-hoc cells against a committed session - there is no live
kernel to attach to. What you have instead:

1. **The run's live output view.** While a commit is executing, its
   "Version" page shows the currently-running cell's stdout, appended as it
   is produced. This cell's `print(..., flush=True)` every `POLL_SECONDS`
   is what populates that view - open the page, scroll to the bottom, close
   it, come back in 20 minutes. This is the mechanism the rest of this
   section is built around.
2. **`monitor_status.txt`**, written into `/kaggle/working` alongside every
   printed snapshot. This is a second, independent channel: the Data/Output
   file browser can preview or download it without opening the run-log
   view at all, and it survives as a persisted output file after the run
   finishes or dies, which the live log view does not guarantee.

**`flush=True` is not decorative.** A plain `print()` inside a long-running
cell can sit in an internal buffer for an unpredictable stretch before it
becomes visible anywhere, which would defeat the entire point of a
periodic-check monitor - you would check in and see stale output. Every
write below is flushed immediately and explicitly, on both channels.


In [ ]:
import json, time, os, sys
from pathlib import Path
from datetime import datetime

COMPONENTS = ["socratic", "code_switching", "grounded_refusal",
              "quiz_generation", "darija_preservation", "reasoning_preservation"]
POLL_SECONDS = 300        # 5 min - adjust to 600 for a 10-min cadence
MAX_HOURS = 11.0          # stay inside the 12h batch budget
STALL_MINUTES = 15

status_path = Path("/kaggle/working/monitor_status.txt")
status_file = open(status_path, "a")

def log(line=""):
    """Write to both check-in channels and flush both immediately.

    stdout goes to the run's live log view; the file is the redundant,
    file-browser-visible channel. Neither is useful for periodic checking
    if it can sit unflushed for an arbitrary stretch, so both are flushed
    on every call rather than relying on default buffering behavior.
    """
    print(line, flush=True)
    status_file.write(line + "\n")
    status_file.flush()
    os.fsync(status_file.fileno())

def proc_alive(proc):
    """True if the PID exists and is still our generator. See markdown above.

    /proc is checked first because on Linux it answers both questions at
    once - existence and identity - without depending on signal-permission
    semantics or on the process being our child.
    """
    try:
        with open(f"/proc/{proc.pid}/cmdline", "rb") as f:
            return b"generate_training_data" in f.read()
    except FileNotFoundError:
        return False          # PID is gone
    except PermissionError:
        return True           # exists, not ours to read
    except OSError:
        pass                  # /proc unavailable - fall through
    try:
        os.kill(proc.pid, 0)
        return True
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    except OSError:
        return False

def snapshot():
    now, out = time.time(), []
    for gpu in PORTS:
        d = Path(f"/kaggle/working/out_full_gpu{gpu}")
        parts = []
        for comp in COMPONENTS:
            prog = d / f"{comp}_raw.progress.jsonl"
            if not prog.exists() or prog.stat().st_size == 0:
                continue
            lines = prog.read_text(encoding="utf-8").strip().splitlines()
            if not lines:
                continue
            last = json.loads(lines[-1])
            raw = d / f"{comp}_raw.jsonl"
            total = sum(1 for _ in open(raw, encoding="utf-8")) if raw.exists() else 0
            parts.append((comp, total, (now - last["ts"]) / 60))
        out.append((gpu, proc_alive(gen_procs[gpu]), parts))
    return out

log(f"=== monitor started {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - "
    f"checking every {POLL_SECONDS}s, budget {MAX_HOURS}h ===")

started = time.time()
while True:
    elapsed_h = (time.time() - started) / 3600
    states = snapshot()
    grand_total = sum(total for _, _, parts in states for _, total, _ in parts)
    log(f"[t+{elapsed_h:5.2f}h] {time.strftime('%H:%M:%S')}  "
        f"grand total: {grand_total}/3000 rows on disk")
    for gpu, alive, parts in states:
        log(f"  GPU{gpu} {'RUNNING' if alive else 'EXITED '}")
        for comp, total, idle in parts:
            flag = "  <-- STALLED" if (idle > STALL_MINUTES and alive) else ""
            log(f"    {comp:24s} {total:4d} rows on disk, idle {idle:5.1f} min{flag}")
    if not any(alive for _, alive, _ in states):
        log("Both generators have exited.")
        break
    if elapsed_h > MAX_HOURS:
        log("Hit MAX_HOURS budget - stopping monitor so packaging cells still run.")
        break
    time.sleep(POLL_SECONDS)

status_file.close()


## 6. Final tallies

Reads what is actually on disk rather than trusting any process's exit
status. This is the authoritative answer to "did the run finish?".


In [ ]:
TARGETS = {"socratic": 400, "code_switching": 350, "grounded_refusal": 350,
           "quiz_generation": 200, "darija_preservation": 100,
           "reasoning_preservation": 100}

grand = 0
for gpu in PORTS:
    d = Path(f"/kaggle/working/out_full_gpu{gpu}")
    print(f"--- GPU{gpu} ---")
    for comp, target in TARGETS.items():
        raw = d / f"{comp}_raw.jsonl"
        n = sum(1 for _ in open(raw, encoding="utf-8")) if raw.exists() else 0
        grand += n
        print(f"  {comp:24s} {n:4d}/{target:4d} {'OK' if n >= target else 'SHORT'}")
print(f"\nGrand total across both shards: {grand} / 3000")


## 7. Merge, dedup, package

Each shard only dedups against itself; cross-shard duplicates survive until
`merge_shards.py` runs. Do this before inspecting results.


In [ ]:
!python -m app.services.merge_shards /kaggle/working/out_full_gpu0 /kaggle/working/out_full_gpu1 --output /kaggle/working/final_export


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/dataset_export", "zip", "/kaggle/working/final_export")
print("Packaged: /kaggle/working/dataset_export.zip")


## 8. Review gate - do not skip

Run the `QUALITY_FLAGS.md` checklist against `final_export/train.jsonl`
before treating this dataset as final: `parse_failures` near 0,
`arabic_script` near 100%, dedup survival, and a native-speaker read of ~50
rows for Socratic pedagogy (explain-then-question) and register.

When reading component summary lines, note that `missing_french`,
`missing_citation`, `parse_failures` and `duplicate_skips` count **rejected
attempts**, not written rows. Every row that reaches disk has already
passed every gate. The v4.1 summary line was misread as "174 of 400 rows
lack French" when it meant "174 attempts were rejected for lacking French
and never became rows".
